# StoreDNA — Pipeline stages

Data flows **top to bottom**. This notebook implements **Stage 4** (highlighted).

```
│  1. Source Data          │
│  2. Curate Data          │
│  3. AI Enrichment        │
│  4. Modality Vectors ◀── Current Step │
│  5. StoreDNA Builder     │
│  6. Vector Index         │
│  7. Business Output      │
```

| Stage | Name | Status |
|-------|------|--------|
| 1 | Source Data | Complete |
| 2 | Curate Data | Complete |
| 3 | AI Enrichment | Complete |
| **4** | **Modality Vectors** | **Current** |
| 5 | StoreDNA Builder | Next |

# Retail Store DNA Builder — Stage 4: Modality Vectors

## What is Stage 4?

Stage 4 collapses **many row-level embeddings** (Stage 3) into **one vector per store per signal type** — the inputs for Stage 5 late fusion.

1. **Embedding modalities** — query Azure AI Search by `modality`, mean-pool `content_vector` per `store_id`.
2. **Structured ops** — aggregate weekly KPIs from `fact_operations_weekly`, z-score normalize across stores.

---

## Inputs & outputs

| Input | Source | Stage 4 action | Output |
|-------|--------|----------------|--------|
| Row embeddings | Azure AI Search (`store-dna-embeddings`) | Mean-pool per store | `{modality}_vectors.npz` |
| `fact_operations_weekly.csv` | Stage 2 curated | KPI aggregation + z-score | `structured_ops_vectors.npz` |
| `dim_store.csv` | Stage 2 curated | Store ordering | `store_modality_index.csv` |

**Output directory:** `data/USA_100_Stores/modality_vectors/`

| Modality | Vector dim | Pooling |
|----------|------------|---------|
| reviews | 3072 | mean + L2 norm |
| news | 3072 | mean + L2 norm |
| reports | 3072 | mean + L2 norm |
| products | 3072 | mean + L2 norm |
| ops_weekly | 3072 | mean + L2 norm |
| structured_ops | 5 KPIs | z-score across stores |

---

## Procedure

1. Load `.env` (Azure AI Search credentials).
2. For each text modality, fetch all documents from the search index and mean-pool vectors by `store_id`.
3. Build structured ops vectors from weekly KPI columns.
4. Write `.npz` files + QC manifest for Stage 5 fusion.

## 1. Setup

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.store_dna_modality_vectors import (
    EMBEDDING_MODALITIES,
    STRUCTURED_MODALITY,
    ModalityVectorConfig,
    pool_modality_from_search,
    run_modality_vectors,
)
from src.store_dna_search import get_index_document_count

load_dotenv(PROJECT_ROOT / ".env")

CURATED_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "curated"
MODALITY_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "modality_vectors"

print("Project root:", PROJECT_ROOT)
print("Curated input:", CURATED_DIR)
print("Modality output:", MODALITY_DIR)

## 2. Configuration

In [ ]:
MAX_STORES = None          # None = all 100 stores; use 5 for a quick test
L2_NORMALIZE_POOLED = True # L2-normalize mean-pooled embedding vectors

config = ModalityVectorConfig.from_env(
    PROJECT_ROOT,
    max_stores=MAX_STORES,
    l2_normalize_pooled=L2_NORMALIZE_POOLED,
)

search_cfg = config.search_config()
print("Search index:", config.search_index_name)
print("Embedding dimensions:", config.embedding_dimensions)
print("Max stores:", config.max_stores or "all")
print("L2 normalize pooled:", config.l2_normalize_pooled)
print("Index document count:", get_index_document_count(search_cfg))

## 3. Preview — pool one modality (reviews)

In [ ]:
preview_ids, preview_vecs, preview_counts = pool_modality_from_search(config, "reviews")
print(f"Stores with review vectors: {len(preview_ids)}")
print(f"Vector shape: {preview_vecs.shape}")
print(f"Sample store {preview_ids[0]}: {preview_counts[preview_ids[0]]} review docs")
print(f"First 5 dims: {preview_vecs[0][:5]}")

## 4. Run full Stage 4 — build all modality vectors

In [ ]:
manifest = run_modality_vectors(CURATED_DIR, MODALITY_DIR, config)
print(json.dumps(manifest, indent=2))

## 5. QC — store × modality coverage

In [ ]:
index_df = pd.read_csv(MODALITY_DIR / "store_modality_index.csv")
coverage = (
    index_df.groupby("modality")
    .agg(stores=("has_vector", "sum"), avg_docs=("doc_count", "mean"), dim=("vector_dim", "first"))
    .reset_index()
)
coverage

In [ ]:
# Load one pooled vector file
reviews_npz = np.load(MODALITY_DIR / "reviews_vectors.npz")
store_ids = reviews_npz["store_ids"]
vectors = reviews_npz["vectors"]
print(f"reviews_vectors.npz: {len(store_ids)} stores × {vectors.shape[1]} dims")
print(f"Non-zero rows: {(vectors != 0).any(axis=1).sum()}")

## 6. Output layout

```
data/USA_100_Stores/modality_vectors/
├── reviews_vectors.npz
├── news_vectors.npz
├── reports_vectors.npz
├── products_vectors.npz
├── ops_weekly_vectors.npz
├── structured_ops_vectors.npz
├── store_modality_index.csv
└── modality_vectors_manifest.json
```

---

## 7. Next steps — Stage 5 (StoreDNA Builder)

| Action | Description |
|--------|-------------|
| Late fusion | Concatenate or weighted-combine all modality vectors per store |
| Output | `store_dna_vector` — one fingerprint per store |
| Index | Upload store-level vectors to Azure AI Search for peer lookup |